# Sit & Goard (2023) on dandiset 000483: rotation direction and arena control (Figures S5, S6)

**Paper.** Sit KK, Goard MJ (2023), *Nature Communications* 14:1992,
[doi:10.1038/s41467-023-37704-5](https://doi.org/10.1038/s41467-023-37704-5); claims file
written from the version of record without its supplement, so Figures S5 and S6 were not read.

**Dataset.** [DANDI:000483](https://dandiarchive.org/dandiset/000483/0.230421.2321), version
0.230421.2321, the only published version (the paper cites none). See `single_cue.ipynb` for the
data check of all files.

| Claim | Here |
|---|---|
| figs5-rotation-direction | tested |
| figs6-arena-control | tested |
| fig5k-visual-cue-only | blocked: no recording shares the visual-only files' ROIs (structure shown below) |

The claims file gives two readings of what the bootstrapped KS test compares: (a) differences in
preferred heading between the two conditions for heading-selective cells, against the same
differences with cells paired at random; (b) the correlation of each cell's tuning curves in the
two conditions, against randomly paired cells. p is one minus the fraction of 1000 bootstrap
iterations in which a two-sample KS test passes at alpha 0.05 (Methods, Statistical
information); in each iteration both samples are drawn with replacement and the random pairing
is drawn afresh (a standard choice). Cells: selective by the two-term test with the centred fit
(see `single_cue.ipynb`) in the motor-driven recording.

In [ ]:
import time
import warnings
from multiprocessing import Pool

import numpy as np
import pandas as pd
from scipy import stats

import rsc

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
started = time.time()
cat = rsc.catalog(full=False)
sub = cat[cat.experiment.isin(["motor_both", "motor_cw", "motor_ccw", "animal_controlled"])].reset_index(drop=True)
recs = rsc.load_many(sub.url.tolist())
with Pool(4) as pool:
    for r in recs:
        r["sp"] = rsc.spikes(rsc.dff(r["F"]), pool)
        del r["F"]
print(f"dandiset {rsc.DANDISET_ID} version {rsc.DANDISET_VERSION}: {len(sub)} files loaded and deconvolved "
      f"in {time.time() - started:.0f} s")

In [ ]:
rows = []
for m, r in zip(sub.itertuples(), recs):
    hc = rsc.clean_heading(r["heading"])
    step = rsc.wrap180(np.diff(hc))
    segs = rsc.trial_segments(r["heading"])
    sign = [np.sign(np.median(np.diff(hc[s]))) for s in segs]
    rows.append(dict(experiment=m.experiment, subject=m.subject, file=m.identifier[:6] + " " + m.identifier.split("_")[-1],
                     frames=m.n_frames, rois=m.n_rois, frac_moving=round(float(np.mean(np.abs(step) > 0.5)), 3),
                     trials_pos=int(np.sum(np.array(sign) > 0)), trials_neg=int(np.sum(np.array(sign) < 0))))
info = pd.DataFrame(rows)
print(info.to_string(index=False))

The animal-controlled files (FR, FM) have few frames in which the chamber turns, so their
tuning curves are built from all turning frames, not from full rotations. The files do not mark
light and dark, so all frames are used.

In [ ]:
def null_boot(real, rand_fn, rng, n_iter=1000, closer="less"):
    passes = 0
    for _ in range(n_iter):
        a = rng.choice(real, len(real))
        b = rng.choice(rand_fn(rng), len(real))
        ks = stats.ks_2samp(a, b)
        side = np.median(a) < np.median(b) if closer == "less" else np.median(a) > np.median(b)
        passes += (ks.pvalue < 0.05) and side
    return 1 - passes / n_iter

def compare(tc1, tc2, cells, rng, label):
    """Readings (a) and (b) for two sets of tuning curves of the same ROIs (ROIs x 60)."""
    idx = np.flatnonzero(cells)
    p1, p2 = rsc.peak_heading(tc1[idx]), rsc.peak_heading(tc2[idx])
    # dual cues: a peak and its opposite are the same cue; distance taken modulo 180
    fold = lambda d: np.abs(rsc.wrap180(2 * d)) / 2
    real_a = fold(p1 - p2)
    rand_a = lambda g: fold(p1 - p2[g.permutation(len(idx))])
    real_b = rsc._rowcorr(tc1[idx], tc2[idx])
    rand_b = lambda g: rsc._rowcorr(tc1[idx], tc2[idx][g.permutation(len(idx))])
    pa = null_boot(real_a, rand_a, rng, closer="less")
    pb = null_boot(real_b[np.isfinite(real_b)], lambda g: rand_b(g)[np.isfinite(rand_b(g))], rng, closer="greater")
    ra = rand_a(rng); rb = rand_b(rng)
    print(f"{label}: n={len(idx)}; (a) median |pref diff| (mod 180) real {np.median(real_a):.0f} deg, random {np.median(ra):.0f} deg, "
          f"bootstrapped KS p = {pa:.3g}; (b) median correlation real {np.nanmedian(real_b):.3f}, random {np.nanmedian(rb):.3f}, "
          f"bootstrapped KS p = {pb:.3g}")
    return dict(n=len(idx), a_real=np.median(real_a), a_rand=np.median(ra), p_a=pa,
                b_real=np.nanmedian(real_b), b_rand=np.nanmedian(rb), p_b=pb)

## figs5-rotation-direction

> The diversity of RSC responses (a second peak in light, as in visual axons, and a single peak in
> the dark, as in ADN axons) persists regardless of the direction of chamber rotation
> (bootstrapped KS test, p = 0.0094, 4 recordings over 2 mice).

The four files that rotate both ways; clockwise and counterclockwise trials within each file.

In [ ]:
both = sub.index[sub.experiment == "motor_both"].tolist()
trials = [rsc.trial_tuning(recs[i]["sp"], recs[i]["heading"]) for i in both]
with Pool(4) as pool:
    sel = rsc.selection_many(trials, "double", 401, pool, centre=True)
rng = np.random.default_rng(402)
tc_pos, tc_neg, cells, mice = [], [], [], []
for i, s in zip(both, sel):
    r = recs[i]
    hc = rsc.clean_heading(r["heading"])
    segs = rsc.trial_segments(r["heading"])
    sign = np.array([np.sign(np.median(np.diff(hc[g]))) for g in segs])
    pos = np.concatenate([g for g, x in zip(segs, sign) if x > 0])
    neg = np.concatenate([g for g, x in zip(segs, sign) if x < 0])
    tc_pos.append(rsc.bin_tuning(r["sp"][pos], hc[pos])); tc_neg.append(rsc.bin_tuning(r["sp"][neg], hc[neg]))
    cells.append(s.is_hd.values); mice.append(np.repeat(sub.subject[i], len(s)))
    print(f"{info.file[i]}: {s.is_hd.sum()} of {len(s)} selective")
tc_pos, tc_neg, cells, mice = np.vstack(tc_pos), np.vstack(tc_neg), np.concatenate(cells), np.concatenate(mice)
res5 = {"selective": compare(tc_pos, tc_neg, cells, rng, "selective cells, 4 files"),
        "all active": compare(tc_pos, tc_neg, (tc_pos.sum(1) > 0) & (tc_neg.sum(1) > 0), rng, "all active cells, 4 files")}
for m in np.unique(mice):
    compare(tc_pos, tc_neg, cells & (mice == m), rng, f"selective cells, {m}")

**Result: reproduced under reading (b).** For the 407 cells selective in the four files that rotate both ways, the correlation of each cell's clockwise and counterclockwise tuning curves (median 0.211) exceeds that of randomly paired cells (0.122), bootstrapped KS p = 0.002. Reading (a), differences in preferred heading, gives p = 0.165 (median 24 against 30 degrees). By mouse, where random pairs come from the same mouse, (b) passes for SKKS125 only (p = 0.046; SKKS126 0.114, SKKS127 0.697). Over all active cells neither reading passes (p = 0.512 and 0.543). The files hold 3 mice where the paper reports 2, and their light condition is not marked, so the light and dark responses the claim names are not separated.

## figs6-arena-control

> The diversity of RSC responses persists whether or not the mouse is in control of the arena
> (bootstrapped KS test, p = 0.031, 7 recordings over 3 mice).

Pairs: each animal-controlled file (FR or FM) with the motor-driven file of the same mouse and
day, which has the same ROIs. Cells selective in the motor-driven file. Tuning in the
animal-controlled file from all frames in which the chamber turns (more than 0.5 degree per
frame).

The result is also shown without any pair whose animal-controlled file visits fewer than half of the 60 heading bins (`bin_tuning` sets an unvisited bin to 0), and for each pair alone, where random pairs come from the same pair of files.

In [ ]:
pairs6 = []
for i in sub.index[sub.experiment == "animal_controlled"]:
    j = sub.index[(sub.experiment != "animal_controlled") & (sub.date == sub.date[i]) & (sub.subject == sub.subject[i])
                  & (sub.n_rois == sub.n_rois[i])]
    if len(j):
        pairs6.append((i, j[0]))
print("pairs (animal-controlled, motor):", [(info.file[i], info.file[j]) for i, j in pairs6])
need = [j for _, j in pairs6]
with Pool(4) as pool:
    sel6 = rsc.selection_many([rsc.trial_tuning(recs[j]["sp"], recs[j]["heading"]) for j in need], "double", 403, pool, centre=True)
tc_a, tc_m, cells6, frames, pair_id = [], [], [], [], []
for (i, j), s in zip(pairs6, sel6):
    hi = rsc.clean_heading(recs[i]["heading"])
    moving = np.r_[False, np.abs(rsc.wrap180(np.diff(hi))) > 0.5]
    tc_a.append(rsc.bin_tuning(recs[i]["sp"][moving], hi[moving]))
    tc_m.append(rsc.trial_tuning(recs[j]["sp"], recs[j]["heading"]).mean(2))
    cells6.append(s.is_hd.values)
    pair_id.append(np.repeat(info.file[i], len(s)))
    occ = np.bincount(np.clip(((hi[moving] + 180) // 6).astype(int), 0, 59), minlength=60)
    frames.append((info.file[i], int(moving.sum()), int((occ > 0).sum())))
    print(f"{info.file[i]} / {info.file[j]}: {s.is_hd.sum()} of {len(s)} selective in the motor file; "
          f"{moving.sum()} turning frames, {int((occ > 0).sum())} of 60 heading bins visited")
tc_a, tc_m, cells6, pair_id = np.vstack(tc_a), np.vstack(tc_m), np.concatenate(cells6), np.concatenate(pair_id)
rng = np.random.default_rng(404)
res6 = {"selective": compare(tc_a, tc_m, cells6, rng, "selective cells, all pairs")}
# without the pairs whose animal-controlled file visits fewer than half of the 60 heading bins
# (bin_tuning sets an unvisited bin to 0, so their curves are mostly zeros)
sparse = [f for f, _, nb_ in frames if nb_ < 30]
res6["without sparse"] = compare(tc_a, tc_m, cells6 & ~np.isin(pair_id, sparse), rng, f"selective cells, without {sparse}")
for f in np.unique(pair_id):
    compare(tc_a, tc_m, cells6 & (pair_id == f), rng, f"selective cells, pair {f} only")
print(f"\nPaper: 7 recordings over 3 mice; here {len(pairs6)} pairs over {sub.subject[[i for i, _ in pairs6]].nunique()} mice")

**Result: not reproduced, resting on leaving out the pair with 6 of 60 heading bins (an assumption).** Over all 5 pairs, reading (b) passes: for 382 cells selective in the motor-driven file, animal-controlled and motor-driven tuning curves correlate more (median 0.026) than randomly paired cells (-0.031), bootstrapped KS p = 0.034; reading (a) gives p = 0.912. That result rests on the pair SKKS125 2022-07-21, whose animal-controlled file visits 6 of 60 heading bins, so its 112 cells have tuning curves that are 0 in the other 54 bins. Without it neither reading passes (n = 270; (a) p = 0.969, (b) p = 0.903, median correlation 0.090 against 0.104 at random), and no pair passes on its own (p 0.781 to 1). The pooled pass comes from random pairs drawn across pairs of files. The dataset holds 5 pairs over 4 mice, where the paper reports 7 recordings over 3.

## fig5k-visual-cue-only (blocked)

> When only the visual cues rotate (no proprioceptive or somatosensory input), the visual
> contributions to tuning are retained, but heading-cell responses are strongly degraded.

The claim needs neurons matched between a standard-rotation recording and a visual-cue-only one
(Figure 5K legend). The dataset's two visual-cue-only ("Suspended") files are listed below with
every file of the same mice, their numbers of ROIs, and their structure in blocks of 1250
frames: the heading step and the share of still frames per block, and the mean raw fluorescence.
A standard rotation inside the same file would show as blocks that differ in some way other than
the direction of rotation.

In [ ]:
same_mice = cat[cat.subject.isin(cat.subject[cat.experiment == "visual_only"])]
print(same_mice[["experiment", "subject", "identifier", "n_frames", "n_rois"]].to_string(index=False))
vis = cat[cat.experiment == "visual_only"].reset_index(drop=True)
vrecs = rsc.load_many(vis.url.tolist())
for m, r in zip(vis.itertuples(), vrecs):
    hc = rsc.clean_heading(r["heading"])
    st = rsc.wrap180(np.diff(hc))
    blocks = range(0, len(st), 1250)
    print(f"\n{m.identifier}: {m.n_frames} frames, {m.n_rois} ROIs, {len(rsc.trial_segments(r['heading']))} trials")
    print("  median heading step per block (deg/frame):", [round(float(np.median(st[k:k + 1250])), 2) for k in blocks])
    print("  share of still frames per block:", [round(float(np.mean(np.abs(st[k:k + 1250]) < 0.5)), 2) for k in blocks])
    print("  mean raw fluorescence per block:", [round(float(r["F"][k:k + 1250].mean()), 0) for k in blocks])
print("\nROI counts shared between a visual-only file and another file of the same mouse:",
      sorted(set(vis.n_rois) & set(same_mice[same_mice.experiment != "visual_only"].n_rois)))

**Result: blocked (metadata_insufficient).** Each visual-cue-only file shares its number of ROIs with no other file of its mouse, so no standard recording holds the same ROIs, and no ROI matching is given. Inside each file the blocks alternate between the two directions of rotation at about 2 degrees per frame, and nothing in the heading or the fluorescence marks some blocks as standard rotation: the files do not say which frames, if any, are standard rotation. The last block of each is still.

## Summary

| Claim | Outcome |
|---|---|
| figs5-rotation-direction | reproduced under reading (b) (p = 0.002); (a) p = 0.165 |
| figs6-arena-control | not reproduced, assumed (the 6-bin pair left out): p = 0.969 and 0.903; with it (b) p = 0.034 |
| fig5k-visual-cue-only | blocked: no recording shares the visual-only files' ROIs |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: Figures S5 and S6 were not read, so what the test compares follows the two readings of the claims file. Spikes are computed without neuropil correction, cells are selected with the centred fit, and light and dark are not separated, as the files do not mark them. For figs6, the pair whose animal-controlled file visits 6 of 60 heading bins is left out, an assumption; the result with it is shown.

What was found about the dataset: the recordings in both directions are over 3 mice (paper 2), and the animal-controlled pairs are 5 over 4 mice (paper 7 over 3). The files do not mark the light condition. The two visual-cue-only files share their ROIs with no other file, and nothing in them marks standard and visual-only rotation.